<a href="https://colab.research.google.com/github/CristovaoDias/LEFA-MECREL/blob/main/Notebooks_Colab/TP01_Analise_Dimensional_e_Graficos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌊 Mecânica e Relatividade (LEFA-MECREL) — 2026/2027
## Caderno Laboratorial TP01: Análise Dimensional, Modelação Numérica e Visualização Científica

**Docente:** Cristóvão de Sousa Dias (`css@isep.ipp.pt`)  
**Departamento de Física (DFI)** — Instituto Superior de Engenharia do Porto (ISEP)  
**Articulação:** Suporte computacional direto à **Ficha TP 1** e aos **Slides TP01**.

---

### 🎯 Objetivos de Aprendizagem Desta Sessão:
1. **Verificação Dimensional Simbólica:** Utilizar o módulo `sympy.physics.units` para verificar dimensões de constantes físicas no computador.
2. **Vetorização Numérica em NumPy:** Abandonar ciclos `for` lentos e calcular vetores físicos de velocidade de tsunami $v(h) = \sqrt{gh}$ instantaneamente.
3. **Rigor Gráfico em Matplotlib:** Produzir gráficos com padrão de publicação científica (unidades nos eixos, anotações de engenharia e grelha de leitura).
4. **Integração Numérica com SciPy:** Resolver o problema real de travessia oceânica onde a profundidade marinha $h(x)$ varia continuamente e a dedução analítica no papel é inviável.
5. **Modelação Interativa com Widgets:** Investigar a resposta física da onda através de controlos deslizantes em tempo real.
6. **Ajuste Numérico de Taylor (Trinity 1945):** Efetuar regressão log-log para extrair a potência nuclear de uma explosão a partir de dados históricos.

---
## 1. Configuração do Ambiente Científico e Padrões Gráficos

Começamos por carregar o ecossistema padrão da física computacional em Python: `numpy`, `matplotlib`, `scipy` e `sympy`.

In [ ]:
# Importação das bibliotecas fundamentais
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate as integrate
import sympy as sp
import sympy.physics.units as u
from sympy.physics.units.systems.si import dimsys_SI

# Configuração de estilo padrão para gráficos científicos de engenharia
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['figure.titlesize'] = 15
plt.rcParams['figure.dpi'] = 100

print("✅ Ambiente configurado com sucesso! Bibliotecas prontas para a aula laboratorial.")

---
## 2. Verificação Dimensional Simbólica com Álgebra Computacional (SymPy)
*(Articulação com o Exercício 1 e Problema 5 da Ficha TP 1)*

Antes de efetuar simulações numéricas, podemos utilizar álgebra simbólica no Python (`sympy`) para efetuar o cancelamento automático de dimensões fundamentais da base $[M, L, T]$.

Vamos validar as deduções da **Ficha TP 1**:
- **Exercício 1 (Newton):** $[G] = \frac{[F][r^2]}{[m_1][m_2]} = [M^{-1} L^3 T^{-2}]$
- **Problema 5 (Stokes):** $[\eta] = \frac{[F_v]}{[r][v]} = [M L^{-1} T^{-1}]$
- **Problema 4 (Tsunami):** $[\sqrt{g \cdot h}] = [L T^{-1}]$

In [ ]:
# Definição das grandezas fundamentais como símbolos algébricos estritamente positivos
M, L, T = sp.symbols('M L T', positive=True)

# 1. Constante de Gravitação Universal de Newton: F = G * m1 * m2 / r^2
# Força: [F] = M * L / T^2
dim_F = M * L / (T**2)
dim_G = (dim_F * L**2) / (M**2)
print(f"1. Equação dimensional de [G] (Newton):   {sp.simplify(dim_G)}   -> [M^-1 L^3 T^-2]")

# 2. Viscosidade Dinâmica da Lei de Stokes: F_v = 6*pi * eta * r * v
dim_v = L / T
dim_eta = dim_F / (L * dim_v)
print(f"2. Equação dimensional de [eta] (Stokes):  {sp.simplify(dim_eta)}     -> [M L^-1 T^-1] (Pa*s)")

# 3. Velocidade do Tsunami em Águas Rasas: v = sqrt(g * h)
dim_g = L / (T**2)
dim_tsunami = sp.sqrt(dim_g * L)
print(f"3. Equação dimensional de sqrt(g*h):       {sp.simplify(dim_tsunami)}         -> [L T^-1] (Velocidade!)")

---
## 3. Modelação do Tsunami em Águas Rasas e Rigor Gráfico
*(Articulação com o Problema 4 da Ficha TP 1 e Slides TP01)*

Pelo método de Rayleigh, deduzimos que a velocidade de propagação de um tsunami em regime de águas rasas (onde o comprimento de onda $\lambda \gg h$) é:
$$v(h) = \sqrt{g \cdot h}$$

Vamos gerar um vetor contínuo de 500 profundidades entre $5\text{ m}$ (zona costeira) e $6000\text{ m}$ (fossas abissais do Pacífico) e traçar o gráfico padrão de engenharia com as unidades físicas nos eixos.

In [ ]:
g = 9.80665  # Aceleração gravítica nominal [m/s^2]

# Geração de 500 profundidades entre 5 m e 6000 m com espaçamento linear
h_profundidades = np.linspace(5, 6000, 500)  # [m]

# Cálculo vetorial da velocidade (sem ciclos for!)
v_ms = np.sqrt(g * h_profundidades)        # [m/s]
v_kmh = v_ms * 3.6                         # Conversão para [km/h]

# Criação da figura com dimensões adequadas a relatório técnico
fig, ax = plt.subplots(figsize=(10, 5.5))

# Traçado da curva física v(h)
ax.plot(h_profundidades, v_kmh, color='#005691', linewidth=2.5, label=r'$v(h) = \sqrt{g \cdot h}$')

# Destaque dos dois regimes operacionais da Ficha TP 1:
# 1. Plataforma continental / Costa rasa (h = 10 m)
# 2. Fundo abissal do Oceano Pacífico (h = 4500 m)
h_costa = 10.0
v_costa = np.sqrt(g * h_costa) * 3.6

h_pacifico = 4500.0
v_pacifico = np.sqrt(g * h_pacifico) * 3.6

# Marcação e anotação dos pontos notáveis
ax.scatter([h_costa], [v_costa], color='#dc2626', s=80, zorder=5)
ax.annotate(f'Costa Rasa ($h=10$ m)\n$v \\approx {v_costa:.1f}$ km/h', 
            xy=(h_costa, v_costa), xytext=(h_costa + 400, v_costa + 80),
            arrowprops=dict(facecolor='#dc2626', shrink=0.08, width=1.5, headwidth=7),
            fontweight='bold', color='#dc2626')

ax.scatter([h_pacifico], [v_pacifico], color='#047857', s=80, zorder=5)
ax.annotate(f'Pacífico Profundo ($h=4500$ m)\n$v \\approx {v_pacifico:.0f}$ km/h (Jato Comercial!)', 
            xy=(h_pacifico, v_pacifico), xytext=(h_pacifico - 2200, v_pacifico - 90),
            arrowprops=dict(facecolor='#047857', shrink=0.08, width=1.5, headwidth=7),
            fontweight='bold', color='#047857')

# Formatação dos eixos com identificação de grandezas e unidades
ax.set_title('Velocidade de Propagação do Tsunami em Função da Profundidade Oceânica', pad=12)
ax.set_xlabel('Profundidade da Coluna de Água, $h$ [m]')
ax.set_ylabel('Velocidade de Propagação, $v$ [km/h]')
ax.set_xlim(0, 6200)
ax.set_ylim(0, 950)
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(loc='lower right', frameon=True)

plt.tight_layout()
plt.show()

---
## 4. Estimativas Rápidas de Ordem de Grandeza (*Problemas de Fermi*)
*(Articulação com o Problema 6 da Ficha TP 1)*

No **Problema 6 da Ficha TP 1**, analisámos o caso histórico do sismo de Prince William (Alasca, 1964) e o alerta emitido para Honolulu (Havai):
- Distância em linha reta: $d = 3800\text{ km}$
- Profundidade média plana: $h = 4500\text{ m}$

Vamos implementar este cálculo de ordem de grandeza e gerar uma resposta operacional automática.

In [ ]:
distancia_km = 3800.0
distancia_m = distancia_km * 1000.0
profundidade_media = 4500.0

# Velocidade média
v_media_ms = np.sqrt(g * profundidade_media)
v_media_kmh = v_media_ms * 3.6

# Tempo de trânsito em regime homogéneo
tempo_seg = distancia_m / v_media_ms
tempo_horas = tempo_seg / 3600.0
horas_inteiras = int(tempo_horas)
minutos_restantes = int((tempo_horas % 1) * 60)

print("="*55)
print("🚨 RELATÓRIO DE ALERTA DE TSUNAMI (ESTIMATIVA DE FERMI)")
print("="*55)
print(f"Distância Alasca -> Havai:       {distancia_km:.0f} km")
print(f"Profundidade Média Assumida:     {profundidade_media:.0f} m")
print(f"Velocidade Média em Alto-Mar:    {v_media_kmh:.1f} km/h ({v_media_ms:.1f} m/s)")
print(f"Janela de Alerta Estimada:       {horas_inteiras} horas e {minutos_restantes} minutos ({tempo_horas:.2f} h)")
print("="*55)

---
## 5. Modelação Realista com Relevo Marinho Variável $h(x)$ e Integração Numérica
*(Articulação direta com a Pergunta c do Problema 6 da Ficha TP 1)*

Num oceano real, o fundo marinho não é plano: varia continuamente com fossas abissais, dorsais submarinas e o talude continental. Nestas condições, a velocidade da onda muda em cada ponto da trajetória: $v(x) = \sqrt{g \cdot h(x)}$.

O tempo total de travessia deixa de ser uma divisão elementar $d / v$ e passa a ser dado pelo **integral de linha**:
$$t_{\text{real}} = \int_0^d \frac{dx}{v(x)} = \int_0^d \frac{dx}{\sqrt{g \cdot h(x)}}$$

Além disso, pelo princípio de conservação do fluxo de energia hidrodinâmica ($\mathcal{F} \propto \rho g v A^2 \approx \text{constante}$), a amplitude da onda amplifica-se na costa segundo a **Lei de Green**:
$$\frac{A(x)}{A_0} = \left( \frac{h_0}{h(x)} \right)^{1/4}$$

Vamos simular este perfil batimétrico real e integrá-lo numericamente usando `scipy.integrate.simpson`.

In [ ]:
# 1. Discretização da rota oceânica Alasca -> Havai em 1000 pontos
x_km = np.linspace(0, 3800, 1000)
x_m = x_km * 1000.0

# 2. Construção de um perfil batimétrico descendente e ascendente realista:
#    - x = 0 km: Plataforma do Alasca (h = 80 m)
#    - x = 200 km: Fossa das Aleutas (h = 5800 m)
#    - x = 200 a 3500 km: Bacia abissal do Pacífico Norte com ondulação batimétrica (h ~ 4200 a 4800 m)
#    - x = 3500 a 3800 km: Subida da cadeia submarina do Havai até à costa de Honolulu (h -> 20 m)
h_perfil = 4400.0 - 500.0 * np.cos(2 * np.pi * x_km / 1200.0)

# Transição suave na costa do Alasca (início)
rampa_alasca = 1.0 - np.exp(-x_km / 80.0)
# Transição na aproximação a Honolulu (fim)
rampa_havai = 1.0 - np.exp(-(3800.0 - x_km) / 60.0)

h_x = (h_perfil * rampa_alasca * rampa_havai) + 20.0  # Mínimo de 20 m junto à praia

# 3. Velocidade local ponto a ponto ao longo da rota
v_x = np.sqrt(g * h_x)  # [m/s]

# 4. Integração numérica do tempo de trânsito: t = integral(1 / v(x) dx)
tempo_real_segundos = integrate.simpson(1.0 / v_x, x=x_m)
tempo_real_horas = tempo_real_segundos / 3600.0

# 5. Fator de amplificação da onda (Lei de Green: A(x) / A0 = (h0 / h(x))^(1/4))
h0_referencia = 4500.0  # Profundidade em mar aberto
fator_shoaling = (h0_referencia / h_x) ** 0.25

# Comparação com a estimativa de Fermi
print("="*60)
print("📊 RESULTADOS DA INTEGRAÇÃO NUMÉRICA COM BATIMETRIA REAL")
print("="*60)
print(f"Tempo de Fermi (Profundidade Média Plana):  {tempo_horas:.2f} h  ({int(tempo_horas)}h {int((tempo_horas%1)*60)}min)")
print(f"Tempo Realista (Integração Numérica h(x)):  {tempo_real_horas:.2f} h  ({int(tempo_real_horas)}h {int((tempo_real_horas%1)*60)}min)")
print(f"Diferença Operacional de Alarme:           {abs(tempo_real_horas - tempo_horas)*60:.1f} minutos")
print(f"Amplificação Máxima na Linha de Costa:     {np.max(fator_shoaling):.1f}x em relação a alto-mar!")
print("="*60)

In [ ]:
# Gráfico Duplo: Perfil do Fundo Oceânico e Consequência Hidrodinâmica
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Painel Superior: Relevo Marinho h(x)
ax1.fill_between(x_km, -h_x, 0, color='#0284c7', alpha=0.35, label='Coluna de Água')
ax1.plot(x_km, -h_x, color='#0369a1', linewidth=2, label='Fundo Marinho $h(x)$')
ax1.axhline(0, color='navy', linestyle='-', linewidth=1.5)
ax1.set_ylabel('Profundidade, $-h(x)$ [m]')
ax1.set_title('Perfil Batimétrico Realista da Rota do Pacífico: Alasca $\to$ Havai (3800 km)')
ax1.set_ylim(-6500, 500)
ax1.annotate('Costa do Alasca\n(Epicentro)', xy=(0, 0), xytext=(80, -2500),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6), fontweight='bold')
ax1.annotate('Honolulu (Havai)\n(Empinamento Costeiro!)', xy=(3800, 0), xytext=(2800, -2500),
             arrowprops=dict(facecolor='#dc2626', shrink=0.05, width=1, headwidth=6), fontweight='bold', color='#dc2626')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='lower center', frameon=True)

# Painel Inferior: Velocidade Local v(x) e Fator de Empinamento
color_v = '#005691'
ax2.plot(x_km, v_x * 3.6, color=color_v, linewidth=2, label='Velocidade da Onda $v(x)$ [km/h]')
ax2.set_xlabel('Distância Percorrida, $x$ [km]')
ax2.set_ylabel('Velocidade, $v(x)$ [km/h]', color=color_v)
ax2.tick_params(axis='y', labelcolor=color_v)
ax2.set_ylim(0, 950)
ax2.grid(True, linestyle='--', alpha=0.6)

# Eixo gémeo à direita para a amplificação de amplitude (Shoaling)
ax2_shoal = ax2.twinx()
color_a = '#dc2626'
ax2_shoal.plot(x_km, fator_shoaling, color=color_a, linestyle='--', linewidth=2, label='Fator de Amplificação $A/A_0$')
ax2_shoal.set_ylabel('Amplificação Relativa da Onda, $A / A_0$', color=color_a)
ax2_shoal.tick_params(axis='y', labelcolor=color_a)
ax2_shoal.set_ylim(0.5, 6.0)
ax2_shoal.grid(False)

plt.tight_layout()
plt.show()

---
## 6. Exploração Interativa com *Widgets* em Sala de Aula

Permite aos alunos modularem a profundidade e a distância percorrida para comparar oceanos (Atlântico vs. Pacífico vs. Mediterrâneo).

In [ ]:
from ipywidgets import interact, FloatSlider

def simular_onda_interativa(profundidade=4000, distancia_km=3800):
    v_ms_calc = np.sqrt(9.80665 * profundidade)
    v_kmh_calc = v_ms_calc * 3.6
    tempo_h = distancia_km / v_kmh_calc
    h_int = int(tempo_h)
    m_int = int((tempo_h % 1) * 60)
    print(f"🌊 SIMULAÇÃO INTERATIVA DE TSUNAMI:")
    print(f"-> Profundidade selecionada:      {profundidade:.0f} m")
    print(f"-> Velocidade de propagação:     {v_kmh_calc:.1f} km/h ({v_ms_calc:.1f} m/s)")
    print(f"-> Tempo de chegada ({distancia_km:.0f} km):    {h_int} horas e {m_int} minutos ({tempo_h:.2f} h)")

# Criação dos sliders interativos para utilização em sala de aula
interact(simular_onda_interativa,
         profundidade=FloatSlider(value=4500, min=20, max=7000, step=100, description='Prof. [m]:'),
         distancia_km=FloatSlider(value=3800, min=100, max=10000, step=100, description='Dist. [km]:'));

---
## 7. Desafio de Fronteira: A Explosão Nuclear Trinity e o Método de Taylor
*(Articulação direta com o Problema 7 da Ficha TP 1 e Slides T01)*

No **Problema 7 da Ficha TP 1**, estudámos como o físico britânico G. I. Taylor decifrou em 1950 a energia da primeira bomba atómica através de fotografias desclassificadas da bola de fogo:
$$R(t) = C \left( \frac{E}{\rho_{\text{ar}}} \right)^{1/5} t^{2/5}$$

Aplicando logaritmos a ambos os membros:
$$\log_{10} R = \frac{2}{5} \log_{10} t + \log_{10} \left[ C \left( \frac{E}{\rho_{\text{ar}}} \right)^{1/5} \right]$$

Num gráfico em escala logarítmica ($\log R$ vs. $\log t$), os pontos experimentais têm de se alinhar rigorosamente numa reta com **declive igual a $2/5 = 0.40$**! Vamos ajustar os dados históricos de Taylor usando regressão linear.

In [ ]:
# Dados históricos das fotografias do teste Trinity (Novo México, 1945)
# Tempo t em segundos, Raio R da bola de fogo em metros
t_dados_s = np.array([0.00038, 0.00080, 0.00150, 0.00300, 0.00600, 0.01000, 0.02500, 0.06200])
R_dados_m = np.array([  25.4,    33.7,    45.0,    61.5,    83.0,   102.0,   130.0,   185.0])

# Transformação logarítmica
log_t = np.log10(t_dados_s)
log_R = np.log10(R_dados_m)

# Regressão linear polinomial de grau 1: log_R = declive * log_t + intercepto
declive, intercepto = np.polyfit(log_t, log_R, 1)

# Cálculo da energia E sabendo que intercepto = log10( C * (E / rho)^(1/5) )
# Admitindo C = 1.0 e densidade do ar rho = 1.20 kg/m^3
rho_ar = 1.20  # kg/m^3
C_taylor = 1.0
termo_k = 10.0 ** intercepto
E_joules = (termo_k / C_taylor)**5 * rho_ar
E_kt_tnt = E_joules / (4.184e12)  # 1 kt TNT = 4.184 x 10^12 J

print("="*55)
print("💥 ANÁLISE HISTÓRICA DO TESTE TRINITY (G. I. TAYLOR, 1950)")
print("="*55)
print(f"Declive Experimental Obtido:       {declive:.4f}  (Teórico: 2/5 = 0.4000)")
print(f"Erro Relativo no Expoente:         {abs(declive - 0.4)/0.4 * 100:.2f}%")
print(f"Energia Calculada por Taylor:      {E_joules:.2e} J")
print(f"Rendimento Nuclear Estimado:       {E_kt_tnt:.1f} kilotons de TNT")
print(f"Valor Oficial Desclassificado:     ~ 20 kt de TNT")
print("="*55)

# Visualização da Regressão Linear Log-Log
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(log_t, log_R, color='#b45309', s=70, zorder=5, label='Dados das Fotografias (Trinity 1945)')

log_t_reta = np.linspace(min(log_t), max(log_t), 100)
log_R_reta = declive * log_t_reta + intercepto
ax.plot(log_t_reta, log_R_reta, color='#0a2540', linewidth=2, 
        label=f'Ajuste: $\log_{{10}} R = {declive:.3f} \log_{{10}} t + {intercepto:.3f}$')

ax.set_title('Validação da Lei de Escala $R(t) \propto t^{2/5}$ da Explosão Nuclear')
ax.set_xlabel('$\log_{10}(t)$ [onde $t$ em segundos]')
ax.set_ylabel('$\log_{10}(R)$ [onde $R$ em metros]')
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

---
## 8. Conexão ao Projeto Integrador CDIO e Desafios para Casa

Nesta primeira aula laboratorial, construíram do zero um fluxo completo de modelação em Engenharia Física:
1. Partiram de uma lei empírica deduzida por **análise dimensional** ($v = \sqrt{gh}$);
2. Implementaram a solução em código vetorial **Python/NumPy**;
3. Representaram os resultados num gráfico profissional com **Matplotlib**;
4. Ultrapassaram as limitações do papel através de **integração numérica** para um relevo marinho real $h(x)$;
5. Ajustaram dados experimentais reais por **regressão log-log** para extrair parâmetros físicos ocultos.

### 📝 Trabalho Autónomo para Esta Semana:
- Concluir os exercícios da **Parte C da Ficha TP 1** (Problema 7 e Problema 8).
- Reunir com a vossa equipa para fechar a escolha de um dos **9 temas do Projeto CDIO**.
- Preparar a leitura prévia do Capítulo 2 do *Serway* (Cinemática Unidimensional).

---
*Instituto Superior de Engenharia do Porto — Departamento de Física — LEFA-MECREL 2026/2027*